## Hands-On Lab 2 - Evaluation

In this lab, you will evaluate the simple models you created in the last lab to see which has the best KPIs (i.e., bias and MAE) to use as a baseline comparison to the machine learning models you will build in later labs. 

Establishing a baseline using a simple model is the first step in a real-world forecasting project. The idea is that if an ML forecasting model can't beat the baseline, you're better off using a simple model.

### Step 0 - Data Splitting

The following code was used to split the Bike Sharing dataset. You won't need this code in the labs, I include it here for completeness:

```python
import pandas as pd
from skforecast.datasets import fetch_dataset

# Fetch dataset from skforecast GitHub repo
bike_sharing = fetch_dataset('bike_sharing', raw = True)

# Set up DataFrame
bike_sharing['date_time'] = pd.to_datetime(bike_sharing['date_time'], format='%Y-%m-%d %H:%M:%S')
bike_sharing = bike_sharing.set_index('date_time')
bike_sharing = bike_sharing.sort_index()

# Split data
end_train = '2012-04-30 23:59:00'
end_validation = '2012-05-07 23:59:00'
end_test = '2012-05-14 23:59:00'

bike_sharing_train = bike_sharing.loc[: end_train, :]
bike_sharing_val   = bike_sharing.loc[end_train:end_validation, :]
bike_sharing_test  = bike_sharing.loc[end_validation:end_test, :]
```

### Step 1 - Loading the Training Data

Run the following code to load the Bike Sharing training data and set the index.

In [ ]:
import pandas as pd

# Load training data
bike_sharing_train = pd.read_csv('BikeSharing_Train.csv', parse_dates = ['date_time'])

# Set index
bike_sharing_train = bike_sharing_train.set_index('date_time')
bike_sharing_train = bike_sharing_train.asfreq('h')
bike_sharing_train = bike_sharing_train.sort_index()

### Step 2 - Loading the Validation Data

Run the following code to load the Bike Sharing validation data, set the index, and display the DataFrame info.

In [ ]:
# Load validation data
bike_sharing_val = pd.read_csv('BikeSharing_Validation.csv', parse_dates = ['date_time'])

# Set index
bike_sharing_val = bike_sharing_val.set_index('date_time')
bike_sharing_val = bike_sharing_val.asfreq('h')
bike_sharing_val = bike_sharing_val.sort_index()

# Get the info
bike_sharing_val.info()

### Step 3 - Chart Set Up

Run the following code to set up line charts using dark mode.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from skforecast.plot import set_dark_theme

# Set fonts/grid defaults
custom_style = {
    'axes.titlesize': 18,
    'axes.labelsize': 15,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'grid.alpha': 0.3,
}

# Make dark mode the default for all plots
set_dark_theme(custom_style = custom_style)  

### Step 4 - Naive Forecasting Model

Run the following code to train a naive forecasting model that predicts the last hour's users (i.e., the first model from Lab #1).

In [ ]:
from skforecast.recursive._forecaster_equivalent_date import ForecasterEquivalentDate

# Forecast horizon
h = 168

# A naive forecasting model using the last hour's users for predictions
naive_last_hour = ForecasterEquivalentDate(offset = 1, n_offsets = 1)

# Learn from the data
naive_last_hour.fit(y = bike_sharing_train['users'])

# Make a lot of predictions so that you can see them in the next step
naive_preds_last_hour = naive_last_hour.predict(steps = h)
naive_preds_last_hour.head()

### Step 5 - Moving Average Model

Run the following code to train an MA(4) forecasting model that uses the moving average of the previous four hours' user count for predictions (i.e., the second model from the last lab).

In [ ]:
import numpy as np

# A MA(4) forecasting model
ma4_forecaster = ForecasterEquivalentDate(offset = 1, n_offsets = 4, agg_func = np.mean)

# Learn from the data
ma4_forecaster.fit(y = bike_sharing_train['users'])

# Make predictions
ma4_preds = ma4_forecaster.predict(steps = h)
ma4_preds.head()

### Step 6 - Revised Naive Forecasting Model

Run the following code to train a naive forecasting model that uses the number of users for the same hour from the previous day (i.e., the third model from the last lab).

In [ ]:
# A naive forecasting model using the previous day's users for the same 
# hour for predictions
naive_last_day = ForecasterEquivalentDate(offset = pd.DateOffset(days = 1), 
                                          n_offsets = 1)

# Learn from the data
naive_last_day.fit(y = bike_sharing_train['users'])

# Make predictions
naive_preds_last_day = naive_last_day.predict(steps = h)
naive_preds_last_day.head()

### Step 7 - Forecast KPI Calculations

Enter and run the following code for the utility function that calculates forecasting model KPIs (i.e., bias and MAE).

In [ ]:
# Enter your lab code here

### Step 8 - Naive Forecast Model KPIs

Enter and run the following code to generate the evaluation metrics, where forecasts are just the previous hour's users.

In [ ]:
# Enter your lab code here

### Step 9 - Visualize the Naive Forecast Model

As covered in the lecture, visualizing forecasts for the validation dataset is a great way to build understanding. Run the following code to visualize the naive model forecasts versus the validation dataset observations.

In [ ]:
# Create a plot 11 inches wide and 5 inches high
fig, ax = plt.subplots(figsize = (11, 5))

# Validation targets
bike_sharing_val['users'].plot(ax = ax, linewidth = 2)

# Forecasts
naive_preds_last_hour.plot(ax = ax, linewidth = 2, linestyle = '--', 
                           label = 'Naive Forecast')

# Display plot
ax.set_title('Naive Model Forecasts for Validation Dataset')
ax.legend()
fig.tight_layout()
plt.show()

### Step 10 - MA(4) Forecast Model KPIs

For the purposes of this lab, you will use an MA(4) model. You will learn techniques later in the course to help you choose the window size. Enter and run the following code to generate the evaluation metrics.

In [ ]:
# Enter your lab code here

### Step 11 - Visualize the MA(4) Model

Run the following code to visualize the MA(4) forecasts versus the validation dataset observations.

In [ ]:
# Create a plot 11 inches wide and 5 inches high
fig, ax = plt.subplots(figsize = (11, 5))

# Validation targets
bike_sharing_val['users'].head(n = h).plot(ax = ax, linewidth = 2)

# Forecasts
ma4_preds.plot(ax = ax, linewidth = 2, linestyle = '--', label = 'MA(4) Forecast')

# Display plot
ax.set_title('MA(4) Model Forecasts for Validation Dataset')
ax.legend()
fig.tight_layout()
plt.show()

### Step 12 - Revised Naive Forecast Model KPIs

Enter and run the following code to generate the evaluation metrics, where forecasts are the users for the same hour of the previous day.

In [ ]:
# Enter your lab code here

### Step 13 - Visualize the Revised Naive Model

Run the following code to visualize the revised naive model forecasts versus the validation dataset observations.

In [ ]:
# Create a plot 11 inches wide and 5 inches high
fig, ax = plt.subplots(figsize = (11, 5))

# Validation targets
bike_sharing_val['users'].head(n = h).plot(ax = ax, linewidth = 2)

# Forecasts
naive_preds_last_day.plot(ax = ax, linewidth = 2, linestyle = '--', 
                          label = 'Revised Naive Forecast')

# Display plot
ax.set_title('Revised Naive Model Forecasts for Validation Dataset')
ax.legend()
fig.tight_layout()
plt.show()